In [0]:
from pyspark.sql.functions import (
    col, to_date, min, max, sum, countDistinct, 
    when, datediff, year, month, dayofmonth, quarter, dayofweek, lit
)

SILVER_TABLE = "workspace.silver_cosmetics.cosmetics_events_silver"
print(f"Đang đọc dữ liệu từ {SILVER_TABLE}...")

silver_df = spark.read.table(SILVER_TABLE)
silver_df = silver_df.withColumn("date_key", to_date(col("event_time")))


# 2.1 DIM_PRODUCT
dim_product = silver_df.select("product_id", "main_category", "sub_category", "brand").dropDuplicates(["product_id"])
(dim_product.write.format("delta").mode("overwrite").saveAsTable("workspace.gold_cosmetics.dim_product"))
print("Đã lưu bảng dim_product")

# 2.2 DIM_USER
purchases = silver_df.filter(col("event_type") == "purchase")
first_purchase_df = purchases.groupBy("user_id").agg(min("date_key").alias("first_purchase_date"))
all_users = silver_df.select("user_id").dropDuplicates()
dim_user = all_users.join(first_purchase_df, on="user_id", how="left")
(dim_user.write.format("delta").mode("overwrite").saveAsTable("workspace.gold_cosmetics.dim_user"))
print("Đã lưu bảng dim_user")

# 2.3 DIM_DATE
dim_date = silver_df.select("date_key").dropDuplicates()
dim_date = (dim_date
    .withColumn("year", year("date_key"))
    .withColumn("month", month("date_key"))
    .withColumn("day", dayofmonth("date_key"))
    .withColumn("quarter", quarter("date_key"))
    .withColumn("day_of_week", dayofweek("date_key"))
    .withColumn("is_weekend", when(col("day_of_week").isin([1, 7]), True).otherwise(False))
)
(dim_date.write.format("delta").mode("overwrite").saveAsTable("workspace.gold_cosmetics.dim_date"))
print("Đã lưu bảng dim_date")


# 3.1 FACT: DAILY PERFORMANCE
fact_daily_performance = (silver_df
    .filter(col("event_type") == "purchase")
    .groupBy("date_key", "brand", "main_category")
    .agg(
        sum("price").alias("total_gmv"),
        countDistinct("user_session").alias("total_orders")
    )
)
(fact_daily_performance.write.format("delta").mode("overwrite").saveAsTable("workspace.gold_cosmetics.fact_daily_performance"))
print("Đã lưu bảng fact_daily_performance")

# 3.2 FACT: USER FUNNEL 
fact_user_funnel = (silver_df
    .groupBy("date_key", "product_id", "user_id") 
    .pivot("event_type", ["view", "cart", "purchase"])
    .count().fillna(0)
    .withColumnRenamed("view", "total_views")
    .withColumnRenamed("cart", "total_add_to_carts")
    .withColumnRenamed("purchase", "total_purchases")
)
(fact_user_funnel.write.format("delta").mode("overwrite").saveAsTable("workspace.gold_cosmetics.fact_user_funnel"))
print("Đã lưu bảng fact_user_funnel")

# 3.3 FACT: RFM SEGMENTATION 
anchor_date_row = silver_df.select(max("date_key")).collect()[0][0]
anchor_date = lit(anchor_date_row)

rfm_base = (silver_df
    .filter(col("event_type") == "purchase")
    .groupBy("user_id")
    .agg(
        datediff(anchor_date, max("date_key")).alias("recency"),
        countDistinct("user_session").alias("frequency"),
        sum("price").alias("monetary")
    )
)

fact_rfm_segmentation = (rfm_base
    .withColumn("calc_date", anchor_date) 
    .withColumn("Customer_Segment", 
        when((col("recency") <= 30) & (col("frequency") >= 3) & (col("monetary") > 50), "VIP")
        .when((col("recency") <= 30) & (col("frequency") == 1), "Newbie")
        .when((col("recency") > 60), "Churning")
        .otherwise("Regular")
    )
)
(fact_rfm_segmentation.write.format("delta").mode("overwrite").saveAsTable("workspace.gold_cosmetics.fact_rfm_segmentation"))
print("Đã lưu bảng fact_rfm_segmentation")

print("HOÀN TẤT TẠO STAR SCHEMA")

Đang đọc dữ liệu từ workspace.silver_cosmetics.cosmetics_events_silver...
Đã lưu bảng dim_product
Đã lưu bảng dim_user
Đã lưu bảng dim_date
Đã lưu bảng fact_daily_performance
Đã lưu bảng fact_user_funnel
Đã lưu bảng fact_rfm_segmentation
HOÀN TẤT TẠO STAR SCHEMA
